# Session 3: LangGraph Orchestration & Workflow Design (Instructor Notebook -- Full Solutions)

This is the **instructor version** of Session 3. It contains the same demos as the student notebook, plus **complete, working solutions** for all four tasks.

## Learning Objectives

By the end of this session, students will be able to:

1. **Create StateGraphs** with typed state schemas
2. **Define nodes and edges** to model workflow steps
3. **Add conditional edges** for dynamic routing based on LLM decisions
4. **Build cyclic workflows** for iterative refinement
5. **Implement a ReAct agent** using the Reason-Act-Observe loop
6. **Add human-in-the-loop** checkpoints for oversight

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Annotated, Literal
import operator
import json
import os

print("All imports successful!")

---
## Demos (Follow Along)

The following five demos are identical to the student notebook.

### Demo 1: LangGraph Basics — Creating a Simple StateGraph

In [ ]:
# Demo 1 - LangGraph Basics

class SimpleState(TypedDict):
    input_text: str
    uppercase_text: str
    word_count: int
    result: str

def uppercase_node(state: SimpleState) -> dict:
    return {"uppercase_text": state["input_text"].upper()}

def count_words_node(state: SimpleState) -> dict:
    return {"word_count": len(state["input_text"].split())}

def format_result_node(state: SimpleState) -> dict:
    return {"result": f"Text: {state['uppercase_text']} | Words: {state['word_count']}"}

graph = StateGraph(SimpleState)
graph.add_node("uppercase", uppercase_node)
graph.add_node("count_words", count_words_node)
graph.add_node("format_result", format_result_node)
graph.set_entry_point("uppercase")
graph.add_edge("uppercase", "count_words")
graph.add_edge("count_words", "format_result")
graph.add_edge("format_result", END)

app = graph.compile()
result = app.invoke({"input_text": "Hello world from LangGraph", "uppercase_text": "", "word_count": 0, "result": ""})
print(f"Result: {result['result']}")

### Demo 2: Adding Conditional Edges for Routing

In [ ]:
# Demo 2 - Conditional Edges for Routing

class RouterState(TypedDict):
    query: str
    category: str
    response: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def classify_node(state: RouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="Classify into exactly one: 'technical', 'creative', or 'factual'. Respond with just the word."),
        HumanMessage(content=state["query"])
    ])
    return {"category": response.content.strip().lower()}

def technical_node(state): 
    r = llm.invoke([SystemMessage(content="You are a technical expert."), HumanMessage(content=state["query"])])
    return {"response": f"[TECHNICAL] {r.content}"}

def creative_node(state):
    r = llm.invoke([SystemMessage(content="You are a creative writer."), HumanMessage(content=state["query"])])
    return {"response": f"[CREATIVE] {r.content}"}

def factual_node(state):
    r = llm.invoke([SystemMessage(content="You are a factual encyclopedia."), HumanMessage(content=state["query"])])
    return {"response": f"[FACTUAL] {r.content}"}

def route_by_category(state):
    c = state["category"]
    if "technical" in c: return "technical"
    elif "creative" in c: return "creative"
    return "factual"

graph = StateGraph(RouterState)
graph.add_node("classify", classify_node)
graph.add_node("technical", technical_node)
graph.add_node("creative", creative_node)
graph.add_node("factual", factual_node)
graph.set_entry_point("classify")
graph.add_conditional_edges("classify", route_by_category, {"technical": "technical", "creative": "creative", "factual": "factual"})
graph.add_edge("technical", END)
graph.add_edge("creative", END)
graph.add_edge("factual", END)
app = graph.compile()

for q in ["How does a hash table work?", "Write a poem about the moon"]:
    r = app.invoke({"query": q, "category": "", "response": ""})
    print(f"Q: {q}\nA: {r['response'][:150]}...\n")

### Demo 3: Building a ReAct Agent with LangGraph

In [ ]:
# Demo 3 - ReAct Agent

class ReActState(TypedDict):
    question: str
    thoughts: list[str]
    actions: list[str]
    observations: list[str]
    final_answer: str
    step_count: int

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def search(query):
    data = {"python creator": "Guido van Rossum created Python in 1991.", "langchain": "LangChain is a framework for LLM applications.", "langgraph": "LangGraph is for stateful multi-step workflows."}
    for key, val in data.items():
        if key in query.lower(): return val
    return f"No results for: {query}"

def think_node(state):
    ctx = "".join(f"Action: {a}\nObs: {o}\n" for a, o in zip(state["actions"], state["observations"]))
    r = llm.invoke([SystemMessage(content="Decide: ACTION: search '<query>' or FINAL ANSWER: <answer>"), HumanMessage(content=f"Q: {state['question']}\n{ctx}")])
    return {"thoughts": state["thoughts"] + [r.content]}

def act_node(state):
    t = state["thoughts"][-1]
    if "FINAL ANSWER:" in t:
        return {"final_answer": t.split("FINAL ANSWER:")[-1].strip(), "step_count": state["step_count"] + 1}
    if "ACTION: search" in t:
        q = t.split("search")[-1].strip().strip("'\"")
        obs = search(q)
        return {"actions": state["actions"] + [f"search('{q}')"], "observations": state["observations"] + [obs], "step_count": state["step_count"] + 1}
    return {"final_answer": t, "step_count": state["step_count"] + 1}

def should_continue(state):
    return "end" if state["final_answer"] or state["step_count"] >= 5 else "think"

graph = StateGraph(ReActState)
graph.add_node("think", think_node)
graph.add_node("act", act_node)
graph.set_entry_point("think")
graph.add_edge("think", "act")
graph.add_conditional_edges("act", should_continue, {"think": "think", "end": END})
app = graph.compile()

result = app.invoke({"question": "Who created Python?", "thoughts": [], "actions": [], "observations": [], "final_answer": "", "step_count": 0})
print(f"Answer: {result['final_answer']}")

### Demo 4: Implementing Cycles for Iterative Refinement

In [ ]:
# Demo 4 - Cycles for Iterative Refinement

class RefinementState(TypedDict):
    topic: str
    draft: str
    feedback: str
    iteration: int
    is_good_enough: bool

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

def write_node(state):
    prompt = f"Improve this text based on feedback.\nText: {state['draft']}\nFeedback: {state['feedback']}\nImproved version (2-3 sentences):" if state["draft"] else f"Write 2-3 sentences about: {state['topic']}"
    r = llm.invoke([HumanMessage(content=prompt)])
    print(f"  [Iter {state['iteration'] + 1}] Draft: {r.content[:80]}...")
    return {"draft": r.content, "iteration": state["iteration"] + 1}

def critique_node(state):
    r = llm.invoke([SystemMessage(content="Rate 1-10 for clarity. If 8+, say 'APPROVED'. Otherwise give feedback."), HumanMessage(content=state["draft"])])
    is_good = "APPROVED" in r.content.upper() or state["iteration"] >= 3
    return {"feedback": r.content, "is_good_enough": is_good}

def should_refine(state): return "end" if state["is_good_enough"] else "write"

graph = StateGraph(RefinementState)
graph.add_node("write", write_node)
graph.add_node("critique", critique_node)
graph.set_entry_point("write")
graph.add_edge("write", "critique")
graph.add_conditional_edges("critique", should_refine, {"write": "write", "end": END})
app = graph.compile()

result = app.invoke({"topic": "Why LangGraph is useful", "draft": "", "feedback": "", "iteration": 0, "is_good_enough": False})
print(f"\nFinal ({result['iteration']} iters): {result['draft']}")

### Demo 5: Human-in-the-Loop (Simulated)

In [ ]:
# Demo 5 - Human-in-the-Loop (Simulated)

class HumanLoopState(TypedDict):
    task: str
    plan: str
    human_approved: bool
    execution_result: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def plan_node(state):
    r = llm.invoke([SystemMessage(content="Create a brief 3-step plan."), HumanMessage(content=state["task"])])
    print(f"  Plan: {r.content[:100]}...")
    return {"plan": r.content}

def human_review_node(state):
    print(f"  [HUMAN REVIEW] Approving plan...")
    return {"human_approved": True}

def execute_node(state):
    r = llm.invoke([SystemMessage(content="Execute this plan."), HumanMessage(content=state["plan"])])
    return {"execution_result": r.content}

graph = StateGraph(HumanLoopState)
graph.add_node("plan", plan_node)
graph.add_node("human_review", human_review_node)
graph.add_node("execute", execute_node)
graph.set_entry_point("plan")
graph.add_edge("plan", "human_review")
graph.add_conditional_edges("human_review", lambda s: "execute" if s["human_approved"] else "end", {"execute": "execute", "end": END})
graph.add_edge("execute", END)
app = graph.compile()

result = app.invoke({"task": "Analyze customer feedback", "plan": "", "human_approved": False, "execution_result": ""})
print(f"Result: {result['execution_result'][:150]}...")

---
## Tasks -- Full Solutions

### Task 1: Build a Simple Linear Workflow with LangGraph

**Approach:** Three sequential nodes: clean (Python string ops), summarize (LLM), translate (LLM). State flows linearly from entry to END.

In [ ]:
# Task 1 - SOLUTION: Simple Linear Workflow

class PipelineState(TypedDict):
    raw_text: str
    clean_text: str
    summary: str
    translation: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

def clean_node(state: PipelineState) -> dict:
    cleaned = " ".join(state["raw_text"].split())
    return {"clean_text": cleaned}

def summarize_node(state: PipelineState) -> dict:
    r = llm.invoke([SystemMessage(content="Summarize in 1-2 sentences."), HumanMessage(content=state["clean_text"])])
    return {"summary": r.content}

def translate_node(state: PipelineState) -> dict:
    r = llm.invoke([SystemMessage(content="Translate to Spanish."), HumanMessage(content=state["summary"])])
    return {"translation": r.content}

graph = StateGraph(PipelineState)
graph.add_node("clean", clean_node)
graph.add_node("summarize", summarize_node)
graph.add_node("translate", translate_node)
graph.set_entry_point("clean")
graph.add_edge("clean", "summarize")
graph.add_edge("summarize", "translate")
graph.add_edge("translate", END)
app = graph.compile()

result = app.invoke({
    "raw_text": "  LangGraph   is a  framework   for building    stateful   multi-step    workflows   using   graphs.  It enables   conditional   routing  and  cycles.  ",
    "clean_text": "", "summary": "", "translation": ""
})
print(f"Clean: {result['clean_text']}")
print(f"Summary: {result['summary']}")
print(f"Spanish: {result['translation']}")

### Task 2: Create a Conditional Routing Agent

**Approach:** A classify node uses the LLM to categorize support messages. Three specialized handlers (billing, technical, general) each use different system prompts. Conditional edges route based on the classification.

In [ ]:
# Task 2 - SOLUTION: Conditional Routing Agent

class SupportState(TypedDict):
    message: str
    category: str
    response: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def classify_support(state: SupportState) -> dict:
    r = llm.invoke([SystemMessage(content="Classify as: 'billing', 'technical', or 'general'. One word only."), HumanMessage(content=state["message"])])
    cat = r.content.strip().lower()
    print(f"  Classified: {cat}")
    return {"category": cat}

def billing_handler(state):
    r = llm.invoke([SystemMessage(content="You are a billing support specialist. Be empathetic and offer solutions."), HumanMessage(content=state["message"])])
    return {"response": f"[BILLING] {r.content}"}

def technical_handler(state):
    r = llm.invoke([SystemMessage(content="You are a technical support engineer. Provide step-by-step troubleshooting."), HumanMessage(content=state["message"])])
    return {"response": f"[TECHNICAL] {r.content}"}

def general_handler(state):
    r = llm.invoke([SystemMessage(content="You are a friendly customer service rep."), HumanMessage(content=state["message"])])
    return {"response": f"[GENERAL] {r.content}"}

def route_support(state):
    if "billing" in state["category"]: return "billing"
    if "technical" in state["category"]: return "technical"
    return "general"

graph = StateGraph(SupportState)
graph.add_node("classify", classify_support)
graph.add_node("billing", billing_handler)
graph.add_node("technical", technical_handler)
graph.add_node("general", general_handler)
graph.set_entry_point("classify")
graph.add_conditional_edges("classify", route_support, {"billing": "billing", "technical": "technical", "general": "general"})
graph.add_edge("billing", END)
graph.add_edge("technical", END)
graph.add_edge("general", END)
app = graph.compile()

for msg in ["I was charged twice", "My app keeps crashing", "What are your business hours?"]:
    r = app.invoke({"message": msg, "category": "", "response": ""})
    print(f"Q: {msg}\nA: {r['response'][:150]}...\n")

### Task 3: Implement a Self-Correcting Code Generation Workflow

**Approach:** The generate node asks the LLM to write code. The test node tries to exec() it. If it fails, the fix node sends the error back to the LLM. A conditional edge loops back until tests pass or max attempts reached.

In [ ]:
# Task 3 - SOLUTION: Self-Correcting Code Generation

class CodeGenState(TypedDict):
    task: str
    code: str
    test_result: str
    passed: bool
    attempts: int

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def generate_code(state: CodeGenState) -> dict:
    if state["code"] and state["test_result"]:
        prompt = f"Fix this code.\n\nCode:\n{state['code']}\n\nError:\n{state['test_result']}\n\nReturn ONLY the corrected Python code, no markdown."
    else:
        prompt = f"Write Python code for: {state['task']}\nInclude a simple test call at the end that prints the result. Return ONLY Python code, no markdown."
    r = llm.invoke([HumanMessage(content=prompt)])
    code = r.content.strip().strip('`').replace('python\n', '', 1)
    print(f"  [Attempt {state['attempts'] + 1}] Generated {len(code)} chars of code")
    return {"code": code, "attempts": state["attempts"] + 1}

def test_code(state: CodeGenState) -> dict:
    try:
        exec(state["code"], {})
        print(f"  [Test] PASSED")
        return {"test_result": "All tests passed", "passed": True}
    except Exception as e:
        print(f"  [Test] FAILED: {e}")
        return {"test_result": str(e), "passed": False}

def check_result(state: CodeGenState) -> str:
    if state["passed"] or state["attempts"] >= 3:
        return "end"
    return "generate"

graph = StateGraph(CodeGenState)
graph.add_node("generate", generate_code)
graph.add_node("test", test_code)
graph.set_entry_point("generate")
graph.add_edge("generate", "test")
graph.add_conditional_edges("test", check_result, {"generate": "generate", "end": END})
app = graph.compile()

result = app.invoke({"task": "Write a function is_prime(n) that returns True if n is prime", "code": "", "test_result": "", "passed": False, "attempts": 0})
print(f"\nFinal code:\n{result['code']}")
print(f"Passed: {result['passed']} (attempts: {result['attempts']})")

### Task 4: Build a Research Agent with Planning and Execution Nodes

**Approach:** The plan node generates a list of research steps. The execute node processes one step per invocation. The check node routes back to execute if more steps remain. Finally, the synthesize node combines all findings.

In [ ]:
# Task 4 - SOLUTION: Research Agent with Planning and Execution

class ResearchState(TypedDict):
    topic: str
    plan: list[str]
    current_step: int
    findings: list[str]
    report: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

def plan_research(state: ResearchState) -> dict:
    r = llm.invoke([
        SystemMessage(content="Create exactly 3 research steps as a JSON list of strings. Example: [\"step1\", \"step2\", \"step3\"]"),
        HumanMessage(content=f"Research topic: {state['topic']}")
    ])
    try:
        plan = json.loads(r.content)
    except:
        plan = ["Research background", "Analyze current state", "Identify future trends"]
    print(f"  Plan: {plan}")
    return {"plan": plan}

def execute_step(state: ResearchState) -> dict:
    step = state["plan"][state["current_step"]]
    r = llm.invoke([
        SystemMessage(content="You are a researcher. Provide a brief finding (2-3 sentences) for this research step."),
        HumanMessage(content=f"Topic: {state['topic']}\nStep: {step}")
    ])
    print(f"  Step {state['current_step'] + 1}: {r.content[:80]}...")
    return {"findings": state["findings"] + [r.content], "current_step": state["current_step"] + 1}

def check_steps(state: ResearchState) -> str:
    if state["current_step"] >= len(state["plan"]):
        return "synthesize"
    return "execute"

def synthesize_report(state: ResearchState) -> dict:
    findings_text = "\n".join(f"{i+1}. {f}" for i, f in enumerate(state["findings"]))
    r = llm.invoke([
        SystemMessage(content="Synthesize these research findings into a coherent summary paragraph."),
        HumanMessage(content=f"Topic: {state['topic']}\n\nFindings:\n{findings_text}")
    ])
    return {"report": r.content}

graph = StateGraph(ResearchState)
graph.add_node("plan", plan_research)
graph.add_node("execute", execute_step)
graph.add_node("check", check_steps)
graph.add_node("synthesize", synthesize_report)
graph.set_entry_point("plan")
graph.add_edge("plan", "execute")
graph.add_edge("execute", "check")
graph.add_conditional_edges("check", check_steps, {"execute": "execute", "synthesize": "synthesize"})
graph.add_edge("synthesize", END)
app = graph.compile()

result = app.invoke({"topic": "The impact of LLMs on software development", "plan": [], "current_step": 0, "findings": [], "report": ""})
print(f"\nResearch Report:\n{result['report']}")

---
## Summary

In this session students learned LangGraph workflow orchestration:

1. **StateGraph Basics** -- Typed state, nodes as functions, edges for control flow.
2. **Conditional Routing** -- Dynamic branching based on LLM decisions.
3. **ReAct Agents** -- The Reason-Act-Observe loop as a cyclic graph.
4. **Iterative Refinement** -- Self-correcting workflows with write-critique-improve cycles.
5. **Human-in-the-Loop** -- Pause points for human oversight.

**Instructor Notes:**
- Emphasize that LangGraph makes control flow *explicit* -- students can draw the graph on a whiteboard.
- For Task 3, discuss the danger of exec() and how production systems would use sandboxed execution.
- For Task 4, note how the plan-execute-check pattern generalizes to any multi-step agent.

**Up Next:** In Session 4, multi-agent architectures -- supervisor-worker, handoffs, and collaborative systems.